# SHADOW — medicul AI pe Colab

Notebook-ul pornește **modelul** (llama-server, pe placa video din Colab) și îi dă o **adresă publică** printr-un tunel.
Serviciul AI de pe laptop (`ai/`, portul 8001) ține documentele NASA și căutarea; la Colab trimite doar promptul gata făcut,
fără nume și fără `crew_id`.

```
aplicația → backend (8000) → serviciul AI pe laptop (8001) → tunel HTTPS → llama-server pe Colab
```

**Pași:** `Runtime → Change runtime type → GPU`, apoi `Runtime → Run all`. La final, celula 6 afișează trei rânduri de pus în
`ai/.env` pe laptop. Lăsați fila deschisă cât durează demo-ul.

**Secrete** (iconița cu cheie din stânga; „Notebook access” pornit):

| Nume | Ce este | Obligatoriu |
|---|---|---|
| `SHADOW_LLM_API_KEY` | parola pe care o cere llama-server; aceeași în `ai/.env` | nu: dacă lipsește, se generează una la fiecare pornire |
| `NGROK_AUTHTOKEN` | tokenul din contul gratuit ngrok | nu: fără el se folosește un tunel Cloudflare, cu adresă nouă la fiecare pornire |
| `NGROK_DOMAIN` | domeniul fix gratuit din ngrok (ex. `shadow-ai.ngrok-free.app`) | nu, dar recomandat: adresa nu se mai schimbă |

Tokenul și cheia nu se scriu niciodată în notebook sau în git.

## 1. Setări

In [ ]:
# Modelul: mic (≈2,7 GB), ca pe un calculator de bord; pe placa din Colab răspunde în 1–2 secunde.
MODEL_REPO = "unsloth/Qwen3.5-4B-GGUF"
MODEL_FILE = "Qwen3.5-4B-Q4_K_M.gguf"
# alternativă: MODEL_REPO = "google/gemma-4-E4B-it-qat-q4_0-gguf"; MODEL_FILE = "gemma-4-E4B_q4_0-it.gguf"  (≈5,2 GB)
MODEL_ALIAS = MODEL_FILE.removesuffix(".gguf")

CONTEXT = 8192        # tokeni: promptul (≈1.500) + istoricul + răspunsul
PORT = 8080
PARALLEL = 2          # câte cereri deodată
# llama.cpp compilat se păstrează în Google Drive, ca a doua pornire să dureze 1 minut, nu 10
CACHE_IN_DRIVE = True

## 2. Placa video și secretele

In [ ]:
import os, secrets, subprocess, time
from google.colab import userdata

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True).stdout or "FĂRĂ GPU: Runtime → Change runtime type → GPU")

def secret(name):
    try:
        return userdata.get(name) or ""
    except Exception:
        return ""

API_KEY = secret("SHADOW_LLM_API_KEY") or secrets.token_urlsafe(24)
NGROK_AUTHTOKEN = secret("NGROK_AUTHTOKEN")
NGROK_DOMAIN = secret("NGROK_DOMAIN")
print("cheie:", "din secrete" if secret("SHADOW_LLM_API_KEY") else "generată acum (se schimbă la fiecare pornire)")
print("tunel:", f"ngrok ({NGROK_DOMAIN or 'adresă aleatoare'})" if NGROK_AUTHTOKEN else "Cloudflare (adresă nouă la fiecare pornire)")

## 3. llama.cpp cu CUDA

Prima dată se compilează (≈5–10 minute, doar pentru placa din această sesiune). Cu `CACHE_IN_DRIVE`, binarul se păstrează în Drive.

In [ ]:
import shutil, pathlib

SERVER = pathlib.Path("/content/llama.cpp/build/bin/llama-server")
CACHE = pathlib.Path("/content/drive/MyDrive/shadow-llama")
gpu_name = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], capture_output=True, text=True).stdout.strip().replace(" ", "_")

if CACHE_IN_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cached = CACHE / gpu_name
    if (cached / "llama-server").exists() and not SERVER.exists():
        SERVER.parent.mkdir(parents=True, exist_ok=True)
        for item in cached.iterdir():
            shutil.copy2(item, SERVER.parent / item.name)
        os.chmod(SERVER, 0o755)
        print("llama-server luat din Drive:", cached)

if not SERVER.exists():
    !git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
    !cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=native -DLLAMA_CURL=OFF -DLLAMA_BUILD_TESTS=OFF -DCMAKE_BUILD_TYPE=Release
    !cmake --build /content/llama.cpp/build --target llama-server -j 4
    if CACHE_IN_DRIVE and SERVER.exists():
        cached = CACHE / gpu_name
        cached.mkdir(parents=True, exist_ok=True)
        for item in SERVER.parent.iterdir():
            if item.is_file() and (item.name == "llama-server" or item.suffix == ".so"):
                shutil.copy2(item, cached / item.name)
        print("llama-server salvat în Drive:", cached)

print("gata:", SERVER.exists())

## 4. Modelul

In [ ]:
from huggingface_hub import hf_hub_download

MODEL_PATH = hf_hub_download(MODEL_REPO, MODEL_FILE, local_dir="/content/models")
print(MODEL_PATH, round(os.path.getsize(MODEL_PATH) / 1e9, 2), "GB")

## 5. Pornirea serverului

Ascultă doar pe `127.0.0.1`; din afară se ajunge la el numai prin tunel, cu cheia API.

In [ ]:
import requests

subprocess.run(["pkill", "-f", "llama-server"], check=False)
LOG = open("/content/llama-server.log", "w")
server = subprocess.Popen([
    str(SERVER), "-m", MODEL_PATH, "--alias", MODEL_ALIAS,
    "--host", "127.0.0.1", "--port", str(PORT), "--api-key", API_KEY,
    "-ngl", "999", "-c", str(CONTEXT), "--parallel", str(PARALLEL),
    "--jinja", "--reasoning-budget", "0",     # fără „gândire” înainte de răspuns: mai repede, mai scurt
    "--no-webui",
], stdout=LOG, stderr=subprocess.STDOUT, env={**os.environ, "LD_LIBRARY_PATH": str(SERVER.parent)})

for _ in range(180):
    try:
        if requests.get(f"http://127.0.0.1:{PORT}/health", timeout=2).status_code == 200:
            break
    except requests.RequestException:
        pass
    if server.poll() is not None:
        break
    time.sleep(1)
print("llama-server:", "pornit" if server.poll() is None else "OPRIT")
print(open("/content/llama-server.log").read()[-1500:])

## 6. Tunelul și rândurile pentru `ai/.env`

In [ ]:
import re

if NGROK_AUTHTOKEN:
    !pip -q install pyngrok
    from pyngrok import conf, ngrok
    conf.get_default().auth_token = NGROK_AUTHTOKEN
    ngrok.kill()
    tunnel = ngrok.connect(PORT, "http", domain=NGROK_DOMAIN) if NGROK_DOMAIN else ngrok.connect(PORT, "http")
    PUBLIC_URL = tunnel.public_url.replace("http://", "https://")
else:
    if not os.path.exists("/content/cloudflared"):
        !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /content/cloudflared
    subprocess.run(["pkill", "-f", "cloudflared"], check=False)
    CF_LOG = "/content/cloudflared.log"
    subprocess.Popen(["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"], stdout=open(CF_LOG, "w"), stderr=subprocess.STDOUT)
    PUBLIC_URL = ""
    for _ in range(60):
        found = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open(CF_LOG).read())
        if found:
            PUBLIC_URL = found.group(0)
            break
        time.sleep(1)

print("Puneți în ai/.env, pe laptop:\n")
print(f"SHADOW_LLM_URL={PUBLIC_URL}")
print(f"SHADOW_LLM_API_KEY={API_KEY}")
print("SHADOW_LLM_MODE=colab_simulation")

## 7. Testul prin tunel

Aceeași cale pe care o folosește serviciul de pe laptop. Ar trebui să dureze 1–3 secunde.

In [ ]:
headers = {"Authorization": f"Bearer {API_KEY}", "ngrok-skip-browser-warning": "1"}
started = time.time()
reply = requests.post(f"{PUBLIC_URL}/v1/chat/completions", headers=headers, timeout=60, json={
    "model": MODEL_ALIAS,
    "messages": [{"role": "user", "content": "In two sentences: why does sleep matter for astronauts?"}],
    "max_tokens": 120, "temperature": 0.3, "chat_template_kwargs": {"enable_thinking": False},
})
print(reply.status_code, f"{time.time() - started:.1f} s")
print(reply.json()["choices"][0]["message"]["content"] if reply.ok else reply.text[:500])
print("fără cheie:", requests.get(f"{PUBLIC_URL}/v1/models", headers={"ngrok-skip-browser-warning": "1"}, timeout=20).status_code, "(trebuie să fie 401)")

## 8. Ține sesiunea trează

Rulați celula în timpul demo-ului (o opriți cu ■). Verifică serverul la fiecare minut și îl repornește dacă a căzut.

In [ ]:
try:
    while True:
        alive = server.poll() is None
        try:
            ok = requests.get(f"http://127.0.0.1:{PORT}/health", timeout=5).status_code == 200
        except requests.RequestException:
            ok = False
        print(time.strftime("%H:%M:%S"), "server OK" if ok else "server NU răspunde", flush=True)
        if not alive:
            print("llama-server s-a oprit: rulați din nou celula 5 (adresa tunelului rămâne aceeași).")
            break
        time.sleep(60)
except KeyboardInterrupt:
    print("oprit")